In [ ]:
# Proposed model: CNN-BiLSTM-Attention, evaluated on the exact same data, features,
# and train/test splits as the Milestone 4 baselines (data/processed/training_sample.parquet)
# so the comparison isolates architecture, not data differences.
import json
import sys
import time
from pathlib import Path

import joblib
import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

sys.path.append(str(Path("..").resolve()))
from src.models.hybrid import build_cnn_bilstm_attention
from src.preprocessing.features import MaxImputer
from src.evaluation.metrics import evaluate, plot_confusion

sns.set_theme(style="whitegrid")
keras.utils.set_random_seed(2026)

In [ ]:
PROCESSED_PATH = Path("../data/processed")
FIGURES_PATH = Path("../reports/figures")
MODELS_PATH = Path("../saved_models")
SEED = 2026
BATCH_SIZE = 1024
MAX_EPOCHS = 25
PATIENCE = 4

features = json.load(open("../reports/final_features.json"))["features"]
label_encoding = json.load(open("../reports/label_encoding.json"))
class_names = sorted(label_encoding, key=label_encoding.get)

data = pd.read_parquet(PROCESSED_PATH / "training_sample.parquet",
                        columns=features + ["Label_Encoded", "split_random", "split_group"])
print(f"rows: {len(data):,}   features: {len(features)}   classes: {len(class_names)}")

In [ ]:
def prepare_split(data, split_col, features, seed):
    """Fit imputer/scaler on the TRAIN rows only, carve a stratified validation
    slice out of train for early stopping (test stays completely untouched),
    and reshape to (samples, features, 1) for the CNN front-end."""
    train_mask = (data[split_col] == "train").to_numpy()
    y = data["Label_Encoded"].to_numpy()

    imputer = MaxImputer().fit(data.loc[train_mask, features])
    scaler = StandardScaler().fit(imputer.transform(data.loc[train_mask, features]))

    def transform(mask):
        X = scaler.transform(imputer.transform(data.loc[mask, features]))
        return X.reshape(X.shape[0], X.shape[1], 1)

    X_train_full, y_train_full = transform(train_mask), y[train_mask]
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.1, stratify=y_train_full, random_state=seed
    )
    X_test, y_test = transform(~train_mask), y[~train_mask]
    return X_tr, y_tr, X_val, y_val, X_test, y_test

In [ ]:
results, per_class_tables, predictions, histories = [], {}, {}, {}

for split_name, split_col in [("random", "split_random"), ("group", "split_group")]:
    print(f"=== {split_name} split ===")
    X_tr, y_tr, X_val, y_val, X_test, y_test = prepare_split(data, split_col, features, SEED)

    class_weight = dict(enumerate(compute_class_weight("balanced", classes=np.arange(len(class_names)), y=y_tr)))

    model = build_cnn_bilstm_attention(len(features), len(class_names))
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    start = time.time()
    history = model.fit(
        X_tr, y_tr, validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE, epochs=MAX_EPOCHS, class_weight=class_weight, verbose=2,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)],
    )
    elapsed = time.time() - start

    y_pred = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).argmax(axis=1)
    summary, per_class = evaluate(y_test, y_pred, class_names)
    epochs_run = len(history.history["loss"])

    results.append({"split": split_name, "model": "CNN-BiLSTM-Attention", **summary,
                     "epochs": epochs_run, "seconds": round(elapsed, 1)})
    per_class_tables[split_name] = per_class
    predictions[split_name] = (y_test, y_pred)
    histories[split_name] = history.history

    model.save(MODELS_PATH / f"cnn_bilstm_attention_{split_name}.keras")
    print(f"[{split_name}] macro-F1={summary['macro_f1']:.4f}  accuracy={summary['accuracy']:.4f}"
          f"  ({epochs_run} epochs, {elapsed/60:.1f} min)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, split_name in zip(axes, histories):
    h = histories[split_name]
    ax.plot(h["loss"], label="train")
    ax.plot(h["val_loss"], label="val")
    ax.set_title(f"{split_name} split")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_PATH / "hybrid_training_curves.png", dpi=150)
plt.show()

In [ ]:
hybrid_results = pd.DataFrame(results)
hybrid_results.to_csv("../reports/hybrid_results.csv", index=False)

hybrid_per_class = pd.DataFrame({split: t["f1"] for split, t in per_class_tables.items()}).round(3)
hybrid_per_class.columns = [f"CNN-BiLSTM-Attention ({c})" for c in hybrid_per_class.columns]
hybrid_per_class.to_csv("../reports/hybrid_per_class_f1.csv")

hybrid_results[["split", "model", "macro_f1", "balanced_accuracy", "accuracy", "epochs", "seconds"]]

In [ ]:
for split_name in histories:
    y_true, y_pred = predictions[split_name]
    plot_confusion(y_true, y_pred, class_names, f"CNN-BiLSTM-Attention -- {split_name} split",
                   FIGURES_PATH / f"confusion_hybrid_{split_name}.png")
    plt.show()

In [ ]:
# Full comparison: our classical baselines (13-class) vs the proposed model (13-class),
# on the same splits. (The base paper's claim and our reproduction of it are a separate,
# 8-class comparison -- see reports/base_paper_reproduction.csv -- kept out of this table
# since it isn't the same classification task.)
baseline = pd.read_csv("../reports/baseline_results.csv").rename(columns={"protocol": "split"})

comparison = pd.concat([
    baseline[baseline["model"] == "XGBoost"][["split", "model", "macro_f1", "accuracy"]],
    hybrid_results[["split", "model", "macro_f1", "accuracy"]],
], ignore_index=True)
comparison["classes"] = 13
comparison.to_csv("../reports/final_comparison.csv", index=False)
comparison